In [49]:
import xarray as xr
import pandas as pd
import logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define the path to the GRIB file
fie_address = '/Volumes/SSD/data/reanalysis-era5-land-temp/'
file_name = fie_address + 'era5-land_all_years.grib'

def inspect_grib_file(file_path):
    if not os.path.exists(file_path):
        logging.error(f"File not found: {file_path}")
        return None
    
    logging.info(f"Opening GRIB file: {file_path}")
    try:
        with xr.open_dataset(file_path, engine='cfgrib') as ds:
            print("--- GRIB File Metadata ---")
            print(f"Dimensions: {ds.dims}")
            print(f"Coordinates: {list(ds.coords.keys())}")
            print(f"Data variables: {list(ds.data_vars.keys())}")
            
            for var in ds.data_vars:
                print(f"\nVariable '{var}':")
                print(f"  Attributes: {ds[var].attrs}")
            
            print("--------------------------")
            return ds
    except Exception as e:
        logging.error(f"Failed to open or inspect GRIB file: {e}")
        return None
ds = inspect_grib_file(file_name)

2025-09-17 11:10:11,885 - INFO - Opening GRIB file: /Volumes/SSD/data/reanalysis-era5-land-temp/era5-land_all_years.grib


--- GRIB File Metadata ---
Dimensions: FrozenMappingWarningOnValuesAccess({'time': 12785, 'step': 24, 'latitude': 42, 'longitude': 53})
Coordinates: ['number', 'time', 'step', 'surface', 'latitude', 'longitude', 'valid_time']
Data variables: ['t2m']

Variable 't2m':
  Attributes: {'GRIB_paramId': 167, 'GRIB_dataType': 'fc', 'GRIB_numberOfPoints': 2226, 'GRIB_typeOfLevel': 'surface', 'GRIB_stepUnits': 1, 'GRIB_stepType': 'instant', 'GRIB_gridType': 'regular_ll', 'GRIB_uvRelativeToGrid': 0, 'GRIB_NV': 0, 'GRIB_Nx': 53, 'GRIB_Ny': 42, 'GRIB_cfName': 'unknown', 'GRIB_cfVarName': 't2m', 'GRIB_gridDefinitionDescription': 'Latitude/Longitude Grid', 'GRIB_iDirectionIncrementInDegrees': 0.1, 'GRIB_iScansNegatively': 0, 'GRIB_jDirectionIncrementInDegrees': 0.1, 'GRIB_jPointsAreConsecutive': 0, 'GRIB_jScansPositively': 0, 'GRIB_latitudeOfFirstGridPointInDegrees': 36.25, 'GRIB_latitudeOfLastGridPointInDegrees': 32.15, 'GRIB_longitudeOfFirstGridPointInDegrees': 75.5, 'GRIB_longitudeOfLastGridPointI

/Users/abhisheksingh/work/witchyFeelings/climate_variable/.venv/lib/python3.13/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [10]:
ds2 = xr.open_dataset(
    file_name,
    engine="cfgrib",
    backend_kwargs={"decode_timedelta": True},
    chunks={"time": 500}   # tune chunk size if needed
)

# Convert to Celsius for easier interpretation
t2m_c = ds["t2m"] - 273.15

In [12]:
t2m_flat = t2m_c.stack(datetime=("time", "step"))
t2m_flat = t2m_flat.assign_coords(valid_time=("datetime", t2m_flat["valid_time"].values))
t2m_flat = t2m_flat.swap_dims({"datetime": "valid_time"})
t2m_flat = t2m_flat.sortby("valid_time")  # make sure sorted

In [16]:
annual_mean = t2m_flat.resample(valid_time="1Y").mean()
annual_max  = t2m_flat.resample(valid_time="1Y").max()
annual_min  = t2m_flat.resample(valid_time="1Y").min()


/Users/abhisheksingh/work/witchyFeelings/climate_variable/.venv/lib/python3.13/site-packages/xarray/groupers.py:509: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/Users/abhisheksingh/work/witchyFeelings/climate_variable/.venv/lib/python3.13/site-packages/xarray/groupers.py:509: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/Users/abhisheksingh/work/witchyFeelings/climate_variable/.venv/lib/python3.13/site-packages/xarray/groupers.py:509: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(


In [17]:
monthly_mean = t2m_flat.resample(valid_time="1M").mean()
monthly_max  = t2m_flat.resample(valid_time="1M").max()
monthly_min  = t2m_flat.resample(valid_time="1M").min()

/Users/abhisheksingh/work/witchyFeelings/climate_variable/.venv/lib/python3.13/site-packages/xarray/groupers.py:509: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
/Users/abhisheksingh/work/witchyFeelings/climate_variable/.venv/lib/python3.13/site-packages/xarray/groupers.py:509: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
/Users/abhisheksingh/work/witchyFeelings/climate_variable/.venv/lib/python3.13/site-packages/xarray/groupers.py:509: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


In [37]:
# 1. Extract month from valid_time
month = t2m_flat["valid_time"].dt.month

# 2. Map months → seasons
def month_to_season(m):
    if m in [12, 1, 2]:
        return "Winter"   # Winter
    elif m in [3, 4, 5]:
        return "Spring"   # Spring
    elif m in [6, 7, 8]:
        return "Summer"   # Summer
    else:
        return "Autumn"   # Autumn

seasons = xr.DataArray(
    [month_to_season(m.item()) for m in month.values],
    coords={"valid_time": t2m_flat["valid_time"]},
    name="season"
)

# 3. Attach season coordinate
t2m_with_season = t2m_flat.assign_coords(season=seasons)

# 4. Extract year from valid_time and attach
years = t2m_with_season["valid_time"].dt.year

# years = t2m_with_season["valid_time"].dt.year
t2m_with_season = t2m_with_season.assign_coords(year=years)

# t2m_with_season = t2m_with_season.assign_coords(year=("valid_time", years))

# 5. Group by year + season
seasonal_mean = t2m_with_season.groupby(["year", "season"]).mean()
seasonal_max  = t2m_with_season.groupby(["year", "season"]).max()


In [38]:
annual_mean_avg = annual_mean.mean(dim=["latitude", "longitude"])
monthly_mean_avg = monthly_mean.mean(dim=["latitude", "longitude"])
seasonal_mean_avg = seasonal_mean.mean(dim=["latitude", "longitude"])


In [39]:
decade1_annual_mean = annual_mean_avg.sel(valid_time=slice("1991", "2000"))
decade2_annual_mean = annual_mean_avg.sel(valid_time=slice("2001", "2010"))
decade3_annual_mean = annual_mean_avg.sel(valid_time=slice("2011", "2020"))
decade4_annual_mean = annual_mean_avg.sel(valid_time=slice("2011", "2024"))


In [40]:
decade4_annual_mean
import xarray as xr
import pandas as pd
import logging
import os
import matplotlib.pyplot as plt
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


In [44]:

# Create output directory if it doesn't exist
if not os.path.exists('output2'):
    os.makedirs('output2')

# Plot (a): Annual Mean Temperature (1991-2020)
plt.figure(figsize=(10, 6))
annual_mean_1991_2020 = annual_mean_avg.sel(valid_time=slice("1991", "2020"))
plt.plot(annual_mean_1991_2020.valid_time.dt.year, annual_mean_1991_2020, marker='o', linestyle='-', label='Annual Mean Temp')
# Add trendline (approximated from image)
z_a = np.polyfit(annual_mean_1991_2020.valid_time.dt.year, annual_mean_1991_2020, 1)
p_a = np.poly1d(z_a)
plt.plot(annual_mean_1991_2020.valid_time.dt.year, p_a(annual_mean_1991_2020.valid_time.dt.year), "r--", label=f'y={z_a[0]:.4f}x + {z_a[1]:.3f}\nR²={0.086:.3f}') # R^2 from image
plt.title('Annual Mean Temperature (1991-2020)')
plt.xlabel('Years')
plt.ylabel('Annual Mean Temperature in °C')
plt.legend()
plt.grid(True)
plt.savefig('output2/annual_mean_temp_1991_2020.png')
plt.close()

# Plot (b): Mean Monthly Temperature (1991-2020)
plt.figure(figsize=(10, 6))
monthly_mean_1991_2020 = monthly_mean_avg.sel(valid_time=slice("1991", "2020"))
# Calculate mean for each month across the years
monthly_mean_by_month = monthly_mean_1991_2020.groupby(monthly_mean_1991_2020.valid_time.dt.month).mean()
plt.plot(monthly_mean_by_month.month, monthly_mean_by_month, marker='o', linestyle='-')
# Add trendline (approximated from image)
z_b = np.polyfit(monthly_mean_by_month.month, monthly_mean_by_month, 1)
p_b = np.poly1d(z_b)
plt.plot(monthly_mean_by_month.month, p_b(monthly_mean_by_month.month), "r--", label=f'y={z_b[0]:.4f}x + {z_b[1]:.3f}\nR²={0.0567:.3f}') # R^2 from image
plt.title('Mean Monthly Temperature (1991-2020)')
plt.xlabel('Month (1991-2020)')
plt.ylabel('Mean Monthly temperature in °C')
plt.xticks(np.arange(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.legend()
plt.grid(True)
plt.savefig('output2/mean_monthly_temp_1991_2020.png')
plt.close()

# Plot (c): Annual mean temperature (1991-2000)
plt.figure(figsize=(10, 6))
plt.plot(decade1_annual_mean.valid_time.dt.year, decade1_annual_mean, marker='o', linestyle='-')
# Add trendline (approximated from image)
z_c = np.polyfit(decade1_annual_mean.valid_time.dt.year, decade1_annual_mean, 1)
p_c = np.poly1d(z_c)
plt.plot(decade1_annual_mean.valid_time.dt.year, p_c(decade1_annual_mean.valid_time.dt.year), "r--", label=f'y={z_c[0]:.4f}x + {z_c[1]:.3f}\nR²={0.2657:.4f}') # R^2 from image
plt.title('Annual mean temperature (1991-2000)')
plt.xlabel('Years')
plt.ylabel('Annual mean temperature in °C')
plt.legend()
plt.grid(True)
plt.savefig('output2/annual_mean_temp_1991_2000.png')
plt.close()

# Plot (d): Annual mean temperature (2001-2010)
plt.figure(figsize=(10, 6))
plt.plot(decade2_annual_mean.valid_time.dt.year, decade2_annual_mean, marker='o', linestyle='-')
# Add trendline (approximated from image)
z_d = np.polyfit(decade2_annual_mean.valid_time.dt.year, decade2_annual_mean, 1)
p_d = np.poly1d(z_d)
plt.plot(decade2_annual_mean.valid_time.dt.year, p_d(decade2_annual_mean.valid_time.dt.year), "r--", label=f'y={z_d[0]:.4f}x + {z_d[1]:.3f}\nR²={0.0078:.4f}') # R^2 from image
plt.title('Annual mean temperature (2001-2010)')
plt.xlabel('Years')
plt.ylabel('Annual mean temperature in °C')
plt.legend()
plt.grid(True)
plt.savefig('output2/annual_mean_temp_2001_2010.png')
plt.close()

# Plot (e): Annual mean temperature (2011-2020)
plt.figure(figsize=(10, 6))
plt.plot(decade3_annual_mean.valid_time.dt.year, decade3_annual_mean, marker='o', linestyle='-')
# Add trendline (approximated from image)
z_e = np.polyfit(decade3_annual_mean.valid_time.dt.year, decade3_annual_mean, 1)
p_e = np.poly1d(z_e)
plt.plot(decade3_annual_mean.valid_time.dt.year, p_e(decade3_annual_mean.valid_time.dt.year), "r--", label=f'y={z_e[0]:.4f}x + {z_e[1]:.3f}\nR²={0.0303:.4f}') # R^2 from image
plt.title('Annual mean temperature (2011-2020)')
plt.xlabel('Years')
plt.ylabel('Annual mean temperature in °C')
plt.legend()
plt.grid(True)
plt.savefig('output2/annual_mean_temp_2011_2020.png')
plt.close()

# Plot (f): Seasonal mean temperature (1991-2000)
plt.figure(figsize=(12, 7))
seasonal_mean_1991_2000_da = seasonal_mean_avg.sel(year=slice("1991", "2000"))

# Convert to DataFrame and pivot
df_seasonal_1991_2000 = seasonal_mean_1991_2000_da.to_dataframe(name='temperature')
df_seasonal_1991_2000_pivot = df_seasonal_1991_2000.reset_index().pivot_table(
    index='year',
    columns='season',
    values='temperature'
)

seasons_order = ['Spring', 'Summer', 'Autumn', 'Winter'] # Spring, Summer, Autumn, Winter
colors = [ "#c44e52",  # crimson
    "#8172b2",  # soft violet
    "#ccb974",  # sand yellow
    "#64b5cd",  # sky blue
]
bar_width = 0.2
years_f = df_seasonal_1991_2000_pivot.index.values
x = np.arange(len(years_f))

for i, season in enumerate(seasons_order):
    if season in df_seasonal_1991_2000_pivot.columns: # Check if season exists
        plt.bar(x + i*bar_width, df_seasonal_1991_2000_pivot[season], bar_width, label=season)

plt.title('Seasonal mean temperature (1991-2000)')
plt.xlabel('Time (1991-2000)')
plt.ylabel('Annual mean temperature in °C')
plt.xticks(x + bar_width * (len(seasons_order) - 1) / 2, years_f)
plt.legend(loc='upper left')
plt.grid(axis='y')
plt.tight_layout()
plt.savefig('output2/seasonal_mean_temp_1991_2000.png')
plt.close()

# Plot (g): Seasonal mean temperature (2001-2010)
plt.figure(figsize=(12, 7))
seasonal_mean_2001_2010_da = seasonal_mean_avg.sel(year=slice("2001", "2010"))
df_seasonal_2001_2010 = seasonal_mean_2001_2010_da.to_dataframe(name='temperature')
df_seasonal_2001_2010_pivot = df_seasonal_2001_2010.reset_index().pivot_table(
    index='year',
    columns='season',
    values='temperature'
)

x = np.arange(len(df_seasonal_2001_2010_pivot.index.values))

for i, season in enumerate(seasons_order):
    if season in df_seasonal_2001_2010_pivot.columns:
        plt.bar(x + i*bar_width, df_seasonal_2001_2010_pivot[season], bar_width, label=season)

plt.title('Seasonal mean temperature (2001-2010)')
plt.xlabel('Time (2001-2010)')
plt.ylabel('Annual mean temperature in °C')
plt.xticks(x + bar_width * (len(seasons_order) - 1) / 2, df_seasonal_2001_2010_pivot.index.values)
plt.legend(loc='upper left')
plt.grid(axis='y')
plt.tight_layout()
plt.savefig('output2/seasonal_mean_temp_2001_2010.png')
plt.close()

# Plot (h): Seasonal mean temperature (2011-2020)
plt.figure(figsize=(12, 7))
seasonal_mean_2011_2020_da = seasonal_mean_avg.sel(year=slice("2011", "2020"))
df_seasonal_2011_2020 = seasonal_mean_2011_2020_da.to_dataframe(name='temperature')
df_seasonal_2011_2020_pivot = df_seasonal_2011_2020.reset_index().pivot_table(
    index='year',
    columns='season',
    values='temperature'
)

x = np.arange(len(df_seasonal_2011_2020_pivot.index.values))

for i, season in enumerate(seasons_order):
    if season in df_seasonal_2011_2020_pivot.columns:
        plt.bar(x + i*bar_width, df_seasonal_2011_2020_pivot[season], bar_width, label=season)

plt.title('Seasonal mean temperature (2011-2020)')
plt.xlabel('Time (2011-2020)')
plt.ylabel('Annual mean temperature in °C')
plt.xticks(x + bar_width * (len(seasons_order) - 1) / 2, df_seasonal_2011_2020_pivot.index.values)
plt.legend(loc='upper left')
plt.grid(axis='y')
plt.tight_layout()
plt.savefig('output2/seasonal_mean_temp_2011_2020.png')
plt.close()

logging.info("All plots generated and saved to the 'output' directory.")


2025-09-17 10:46:17,327 - INFO - All plots generated and saved to the 'output' directory.


In [45]:
import pymannkendall as mk

In [50]:
# Function to apply Mann-Kendall test to a time series
def apply_mann_kendall(time_series):
    if time_series.isnull().all(): # Handle cases with all NaNs
        return np.nan, np.nan
    try:
        # pymannkendall expects a 1D array or list
        # Corrected function call based on user feedback: mk.original_test(x) returns slope and p-value
        result = mk.original_test(time_series.values)
        return result.slope, result.p
    except Exception as e:
        logging.error(f"Error applying Mann-Kendall test: {e}")
        return np.nan, np.nan


# Apply Mann-Kendall test to annual mean temperature for each lat/lon point
logging.info("Applying Mann-Kendall test to annual mean temperature data...")

# We need to iterate over latitude and longitude to apply the test to each time series
# t2m_c has dimensions (valid_time, latitude, longitude)
# We want to apply the test along the 'valid_time' dimension for each lat/lon pair.

# Create empty DataArrays to store trend and p-value
trend_da = xr.DataArray(
    np.full((len(t2m_c['latitude']), len(t2m_c['longitude'])), np.nan),
    coords=[t2m_c['latitude'], t2m_c['longitude']],
    dims=['latitude', 'longitude'],
    name='trend'
)
p_value_da = xr.DataArray(
    np.full((len(t2m_c['latitude']), len(t2m_c['longitude'])), np.nan),
    coords=[t2m_c['latitude'], t2m_c['longitude']],
    dims=['latitude', 'longitude'],
    name='p_value'
)

# Iterate over latitude and longitude
for i, lat in enumerate(t2m_c['latitude'].values):
    for j, lon in enumerate(t2m_c['longitude'].values):
        ts = t2m_c.sel(latitude=lat, longitude=lon)
        trend, p_value = apply_mann_kendall(ts)
        trend_da[i, j] = trend
        p_value_da[i, j] = p_value

logging.info("Mann-Kendall test applied. Generating trend maps...")

# Plotting the Mann-Kendall trend results
# Plotting Trend Map
plt.figure(figsize=(12, 8))
# Use p-value to mask non-significant trends if desired, or plot all trends
# For simplicity, let's plot all trends first. We can add masking later.
trend_map = plt.pcolormesh(trend_da.longitude, trend_da.latitude, trend_da, cmap='coolwarm', shading='auto')
plt.colorbar(trend_map, label='Annual Temperature Trend (°C/year)')
plt.title('Mann-Kendall Trend Analysis: Annual Mean Temperature Trend')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.savefig('output/mk_trend_annual_mean_temp.png')
plt.close()

# Plotting P-value Map (e.g., showing significance)
plt.figure(figsize=(12, 8))
# A common threshold for significance is p < 0.05
# We can visualize this by coloring points where p < 0.05 differently, or by plotting p-values directly.
# Let's plot p-values directly for now.
p_map = plt.pcolormesh(p_value_da.longitude, p_value_da.latitude, p_value_da, cmap='viridis', shading='auto')
plt.colorbar(p_map, label='P-value')
plt.title('Mann-Kendall Trend Analysis: P-value')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.savefig('output/mk_p_value_annual_mean_temp.png')
plt.close()

logging.info("Mann-Kendall trend maps generated and saved to the 'output' directory.")

logging.info("All tasks completed.")


2025-09-17 11:38:38,535 - INFO - Applying Mann-Kendall test to annual mean temperature data...
2025-09-17 11:38:38,905 - ERROR - Error applying Mann-Kendall test: too many indices for array: array is 1-dimensional, but 2 were indexed
2025-09-17 11:38:38,911 - ERROR - Error applying Mann-Kendall test: too many indices for array: array is 1-dimensional, but 2 were indexed
2025-09-17 11:38:38,917 - ERROR - Error applying Mann-Kendall test: too many indices for array: array is 1-dimensional, but 2 were indexed
2025-09-17 11:38:38,923 - ERROR - Error applying Mann-Kendall test: too many indices for array: array is 1-dimensional, but 2 were indexed
2025-09-17 11:38:38,929 - ERROR - Error applying Mann-Kendall test: too many indices for array: array is 1-dimensional, but 2 were indexed
2025-09-17 11:38:38,935 - ERROR - Error applying Mann-Kendall test: too many indices for array: array is 1-dimensional, but 2 were indexed
2025-09-17 11:38:38,942 - ERROR - Error applying Mann-Kendall test: too